# 온라인 제품 판매량 예측 모델
이 노트북은 제공된 데이터를 바탕으로 온라인 제품의 향후 판매량을 예측하는 딥러닝 모델(LSTM)을 구축하는 과정을 담고 있습니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic' # 한글 폰트 설정 (Windows 기준)
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터 로드

In [ ]:
# 데이터 경로 설정
train_path = './data/train.csv'
sales_path = './data/sales.csv'
product_info_path = './data/product_info.csv'
brand_keyword_path = './data/brand_keyword_cnt.csv'
sample_submission_path = './data/sample_submission.csv'

# 데이터 로드
train = pd.read_csv(train_path)
sales = pd.read_csv(sales_path)
product_info = pd.read_csv(product_info_path)
brand_keyword = pd.read_csv(brand_keyword_path)
sample_submission = pd.read_csv(sample_submission_path)

print("Train Data Shape:", train.shape)
print("Sales Data Shape:", sales.shape)
print("Product Info Shape:", product_info.shape)

## 2. EDA (탐색적 데이터 분석)

In [ ]:
# 결측치 확인
print("Train Nulls:\n", train.isnull().sum().sum())
print("Product Info Nulls:\n", product_info.isnull().sum())

# 시간 흐름에 따른 전체 판매량 트렌드 시각화
# 시계열 데이터가 컬럼으로 나열되어 있으므로 melt를 통해 변환
date_columns = train.columns[6:] # ID, 제품코드, 대/중/소/브랜드 이후가 날짜
total_sales_per_day = train[date_columns].sum()

plt.figure(figsize=(15, 6))
plt.plot(total_sales_per_day.index, total_sales_per_day.values)
plt.title('Daily Total Sales Trend')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.xticks(total_sales_per_day.index[::30], rotation=45)
plt.show()

## 3. 데이터 전처리 및 특성 공학

In [ ]:
# 범주형 변수 인코딩
le = LabelEncoder()
train['brand_encoded'] = le.fit_transform(train['브랜드'])

# 데이터 구조 변경 (Wide to Long)
# 시계열 모델링을 위해 날짜별 행으로 변환합니다.
train_melted = train.melt(id_vars=['ID', '제품', '대분류', '중분류', '소분류', '브랜드', 'brand_encoded'], 
                          var_name='date', value_name='sales')

train_melted['date'] = pd.to_datetime(train_melted['date'])

# 날짜 기반 특성 생성
train_melted['month'] = train_melted['date'].dt.month
train_melted['day'] = train_melted['date'].dt.day
train_melted['dayofweek'] = train_melted['date'].dt.dayofweek

print(train_melted.head())

## 4. LSTM 모델을 위한 데이터 준비 (Lag 특성 생성)

In [ ]:
# 과거 판매량 데이터를 활용하기 위해 Lag 데이터 생성
# 여기서는 간단하게 과거 7일간의 데이터를 사용하도록 구성합니다.
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:(i + seq_length)])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

# 예시로 특정 제품 하나에 대해서만 스케일링 및 데이터 분할
# 전체 제품에 대해 수행하려면 반복문 또는 배치 처리가 필요합니다.
target_product_id = train_melted['ID'].unique()[0]
subset = train_melted[train_melted['ID'] == target_product_id][['sales']].values

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(subset)

seq_length = 30 # 과거 30일을 보고 다음날 예측
X, y = create_sequences(scaled_data, seq_length)

# Train/Test Split
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

## 5. 모델 정의 및 학습

In [ ]:
model = Sequential([
    LSTM(64, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True),
    Dropout(0.2),
    LSTM(32, activation='relu'),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

history = model.fit(X_train, y_train, epochs=20, batch_size=32, 
                    validation_data=(X_test, y_test), verbose=1)

## 6. 평가 및 시각화

In [ ]:
# Loss 그래프
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.legend()
plt.show()

# 예측 및 RMSE 평가
predictions = model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, predictions)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

## 7. 결과 제출용 예측

In [ ]:
# 실제 예측 시에는 마지막 30일 데이터를 사용하여 향후 21일(Submission 기간)을 예측합니다.
# 여기서는 예시로 submission 양식에 0을 채워 저장하는 코드를 작성합니다.

final_submission = sample_submission.copy()
print(final_submission.head())

# 실제 예측값을 채우는 로직이 이곳에 포함되어야 함 (생략)
# final_submission.iloc[:, 1:] = predicted_values

final_submission.to_csv('submission_result.csv', index=False)
print("Submission file saved!")